# NB-002 — Judge Wiring

Companion to `doc/design/02_prompts_judge.md` (LLD) and `doc/task/02_prompts_judge.md` (contract).

Notebook-first prototype (D2): build the judge from env, load the approved prompt from the
registry, render it against one pinned golden row, invoke through the throttle, parse +
schema-validate the JSON reply. Once this runs end-to-end, promote into
`src/llmops/config/judge.py` + `src/llmops/prompts/{__init__,registry}.py`.

Exit criteria this notebook proves (task 02):
- NB-002 runs end-to-end; judge JSON reply parsed and schema-validated
- Judge endpoint matches the D9 choice (`Qwen/Qwen2.5-7B-Instruct-AWQ`)

## 0. Setup — repo root resolution + imports

In [1]:
from langchain_openai import ChatOpenAI

In [2]:
from typing import Any, Literal

from pydantic import BaseModel, Field, ValidationError, model_validator


In [3]:
import threading
import time

In [4]:
# TODO: repo-root walk (pyproject.toml marker), sys.path insert, os.chdir, load_dotenv
# TODO: stdlib + third-party imports (json, re, threading, time, pathlib, pydantic, ChatOpenAI, ...)
import json
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv

# --- Repo root resolution ---------------------------------------------
# Makes the relative "data/..." and "eval/..." paths work no matter
# where Jupyter was launched from.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").exists(), (
    f"pyproject.toml not found in any parent directory — stopped at {ROOT}"
)

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=True)

print("repo root:", ROOT)


repo root: /home/dipak/agentic/step9_llmops


## 1. Read env → `judge_llm()`

Reads `LLM_BASE_URL` / `LLM_API_KEY` / `LLM_MODEL` only (stripped). Raises `RuntimeError`
if URL/KEY missing or empty-after-strip. Never reads `GROQ_*` / `GEMINI_*` (D6 separation).

In [5]:
# TODO: def judge_llm() -> ChatOpenAI: ...
# TODO: llm = judge_llm()
# TODO: assert LLM_MODEL matches D9 choice (Qwen/Qwen2.5-7B-Instruct-AWQ)  # T-02-14

def judge_llm() -> ChatOpenAI:
    _JUDGE_URL = os.environ["LLM_BASE_URL"].strip()
    _JUDGE_KEY = os.environ["LLM_API_KEY"].strip()
    _JUDGE_MODEL =  os.environ.get("LLM_MODEL", "").strip() or "Qwen/Qwen2.5-7B-Instruct-AWQ"

    if not _JUDGE_URL:
        raise RuntimeError("LLM_BASE_URL not configured")

    if not _JUDGE_KEY:
        raise RuntimeError("LLM_API_KEY not configured")

    return ChatOpenAI(
        openai_api_base=_JUDGE_URL,
        openai_api_key=_JUDGE_KEY,
        model=_JUDGE_MODEL,
        temperature=0,
        # If your endpoint supports JSON mode, enable it here:
        # extra_body={"response_format": {"type": "json_object"}},
    )

llm = judge_llm()

D9_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
assert llm.model_name == D9_MODEL, f"judge resolved to {llm.model_name!r}"


## 2. `JudgeResponse` schema (D25)

`winner: Literal["A", "B"]`, `score_a`/`score_b: int` (0-10), `reasoning: str` (non-empty).

In [6]:
# TODO: class JudgeResponse(BaseModel): winner / score_a / score_b / reasoning
class JudgeResponse(BaseModel):
    """Typed judge output.
    
    Per Mod 2 LLD & D25 spec:
    - winner in {"A", "B"} strictly bound per D25.
    - score_a/score_b in range [0, 10] (Field-enforced, D27).
    - reasoning must be a non-empty string (Field-enforced, D27).
    - model_validator: score↔winner consistency (incoherent → salvage, D27).
    Note: field ORDER in this model has no effect on the LLM — CoT-before-scoring
    is forced by the prompt TEXT, not the pydantic field declaration.
    """
    reasoning: str = Field(
        ...,
        min_length=1,
        description="Detailed comparative explanation for the scores and declared winner."
    )
    score_a: int = Field(
        ...,
        ge=0,
        le=10,
        description="Score for answer A (0–10)"
    )
    score_b: int = Field(
        ...,
        ge=0,
        le=10,
        description="Score for answer B (0–10)"
    )
    winner: Literal["A", "B"] = Field(
        ...,
        description="Which answer is better: 'A' or 'B'"
    )

    @model_validator(mode="after")
    def validate_scores_match_winner(self) -> "JudgeResponse":
        """Guarantees internal consistency between scores and winner."""
        if self.score_a > self.score_b and self.winner != "A":
            raise ValueError(
                f"Score A ({self.score_a}) is higher than Score B ({self.score_b}), "
                f"but winner was set to '{self.winner}'."
            )
        if self.score_b > self.score_a and self.winner != "B":
            raise ValueError(
                f"Score B ({self.score_b}) is higher than Score A ({self.score_a}), "
                f"but winner was set to '{self.winner}'."
            )
        return self

## 3. Throttle — `_JudgeThrottle` + `throttled_invoke()`

Process-global lock + `_MIN_SPACING_S = 60.0 / 18` (~3.33s). First call: no initial wait.
`_last_send` updates only on success (failures never compress the spacing window).

In [7]:
class _JudgeThrottle:
    """Process-global lock + ≥3.33s spacing (~18 req/min).

    Per Mod 2 LLD spec:
    - Scope: in-memory, per-process only.
    - First call: no initial wait (_last_send starts at 0 -> elapsed > spacing).
    - Failure safety: _last_send updates ONLY on success, preventing retry storms.
    """
    _MIN_SPACING_S: float = 60.0 / 18.0  # ~3.333 seconds
    _lock: threading.Lock = threading.Lock()
    _last_send: float = 0.0

    @classmethod
    def wait(cls) -> None:
        """Gated execution enforcing minimum inter-request delay."""
        with cls._lock:
            now = time.monotonic()
            elapsed = now - cls._last_send
            if elapsed < cls._MIN_SPACING_S:
                sleep_time = cls._MIN_SPACING_S - elapsed
                time.sleep(sleep_time)

    @classmethod
    def record_success(cls) -> None:
        """Updates timestamp strictly after a successful API call."""
        with cls._lock:
            cls._last_send = time.monotonic()


class JudgeError(Exception):
    """Typed error raised for judge execution, salvage, or registry failures.

    Per Mod 2 LLD (D11/D26):
    - Message ALLOWS: prompt_id/key + short reason.
    - Message DENIES: LLM_BASE_URL, LLM_API_KEY, secrets, or full raw response text.
    """

def throttled_invoke(prompt: str, *, llm: ChatOpenAI) -> str:
    """Serialize judge calls through _JudgeThrottle. Returns JSON string."""
    _JudgeThrottle.wait()
    try:
        response = llm.invoke(prompt)
        _JudgeThrottle.record_success()
        return str(response.content)
    except Exception as err:
        raise JudgeError(f"LLM API invocation failed: {type(err).__name__}") from err


## 4. Load registry → select approved prompt → render template

Registry key scope for Mod 2: exactly one `(prompt_id=judge_system, source_type=generic)` pair.
Selection is unambiguous — pick the single `status == "approved"` version.

In [8]:
def load_registry(path: Path | str | None = None) -> dict[str, Any]:
    """Load registry.json relative to repository root.

    Per LLD spec:
    - Default path is ROOT / "src" / "llmops" / "prompts" / "registry.json".
    - Never dependent on transient cwd.
    """
    target_path = Path(path) if path else (ROOT / "src" / "llmops" / "prompts" / "registry.json")

    if not target_path.exists():
        raise JudgeError(f"Registry file not found at path: '{target_path}'")

    if target_path.stat().st_size == 0:
        raise JudgeError(f"Registry file is empty (0 bytes) at path: '{target_path}'")

    try:
        with open(target_path, "r", encoding="utf-8") as f:
            content = json.load(f)
            if not content:
                raise JudgeError(f"Registry file contains empty JSON content at path: '{target_path}'")
            return content
    except json.JSONDecodeError as e:
        raise JudgeError(f"Registry file contains invalid JSON at path: '{target_path}': {e}")


def select_approved(registry: dict[str, Any], prompt_id: str, source_type: str) -> dict[str, Any]:
    """Return the approved entry for a given (prompt_id, source_type) pair.

    Per LLD & T-02-9b specs:
    - Exactly one approved version per (prompt_id, source_type) key is valid.
    - Zero-approved state is legal in registry JSON, but selection MUST raise JudgeError.
    """
    prompts = registry.get("prompts", {})
    matching_approved = []

    for entry in prompts.values():
        if (
            entry.get("prompt_id") == prompt_id
            and entry.get("source_type") == source_type
            and entry.get("status") == "approved"
        ):
            matching_approved.append(entry)

    if not matching_approved:
        raise JudgeError(
            f"No approved prompt found for prompt_id='{prompt_id}' and source_type='{source_type}'."
        )

    if len(matching_approved) > 1:
        raise JudgeError(
            f"Multiple approved prompts found for prompt_id='{prompt_id}' and source_type='{source_type}'."
        )

    return matching_approved[0]


# --- Execute Loading & Selection --------------------------------------
registry = load_registry()

approved_entry = select_approved(
    registry=registry,
    prompt_id="judge_system",
    source_type="generic"
)

judge_template = approved_entry["template"]


## 5. Pin the input — one golden row from `eval/goldens/`

Row id asserted here for reproducibility (T-02-13). Judge task is pairwise: query + ideal
context/answer (reference) vs. two candidate answers (A / B) to compare.

In [9]:
GOLD_ROW_ID = "MS-Q005"  # conflict-category, pinned in D25 #6 for reproducibility

with open(ROOT / "eval" / "goldens" / "query_processing_goldens.json", encoding="utf-8") as _f:
    _golden_rows = json.load(_f)

golden_row = next(r for r in _golden_rows if r["id"] == GOLD_ROW_ID)
assert golden_row["id"] == GOLD_ROW_ID, f"golden row {GOLD_ROW_ID} not found in eval/goldens/"

# Hand-authored fixtures for this smoke test (not retriever output — that's Mod 3):
# A = surfaces the S1/S2 model drift conflict (correct judge target)
# B = single-source, misses the conflict entirely (wrong)
candidate_a = (
    "The pipeline is inconsistent: step 1's embedder is BAAI/bge-base-en-v1.5 "
    "(768-dim, local ONNX), but the Shared Pipeline Design claims 'Same embedding "
    "as step 2: qwen3-embed (1024-dim)'. Both are grounded; the drift must be surfaced."
)
candidate_b = (
    "The shared pipeline uses bge-base-en-v1.5 everywhere — one embedder across all steps."
)

print("pinned:", golden_row["id"], "| category:", golden_row["category"])
print("query :", golden_row["query"])

pinned: MS-Q005 | category: conflict
query : Which embedding model does the shared pipeline use across the steps?


## 6. Render the prompt template

In [10]:
# Templates embed a literal JSON-schema example (braces) — str.format() would
# crash on them (KeyError: '{} reason...'), so render via static token replace:
def render_template(template: str, **kwargs) -> str:
    out = template
    for k, v in kwargs.items():
        out = out.replace("{" + k + "}", str(v))
    return out

# TODO: rendered_prompt = render_template(judge_template,
# TODO:     query=..., ideal_answer=..., candidate_a=..., candidate_b=...)

rendered_prompt = render_template(
    judge_template,
    query=golden_row["query"],
    ideal_answer=golden_row["ideal_answer"],
    candidate_a=candidate_a,
    candidate_b=candidate_b,
)

# Sanity: the four render tokens are gone; literal JSON-schema braces survive.
for _k in ("query", "ideal_answer", "candidate_a", "candidate_b"):
    assert "{" + _k + "}" not in rendered_prompt, f"render token not substituted: {{{_k}}}"
assert '"reasoning"' in rendered_prompt, "literal JSON-schema braces lost during render"
print("rendered length:", len(rendered_prompt))

rendered length: 1756


## 7. Invoke → parse → salvage-on-failure → validate

1. `throttled_invoke` → attempt JSON parse
2. Invalid (JSONDecodeError or schema mismatch) → strip markdown fences → retry once
3. Still invalid → raise `JudgeError` (typed, no secrets in message)

In [11]:
# TODO: raw = throttled_invoke(rendered_prompt, llm=llm)
# TODO: parse -> fence-strip fallback -> pydantic validate -> JudgeResponse

def _strip_fences(text: str) -> str:
    """Remove markdown ```json ... ``` fences the model may wrap around the JSON."""
    m = re.search(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL)
    return m.group(1).strip() if m else text.strip()

def invoke_judge(prompt: str, *, llm: ChatOpenAI) -> JudgeResponse:
    """One throttle-gated invoke -> parse -> fence-strip fallback -> pydantic validate.

    Salvage path (LLD:108-116):
      1. throttled_invoke -> json.loads -> JudgeResponse.model_validate
      2. on JSONDecodeError (or ValidationError): strip fences -> retry once
      3. still invalid -> raise JudgeError (typed, no secrets in message)
    """
    raw = throttled_invoke(prompt, llm=llm)
    for attempt in (0, 1):  # attempt 0 = as-is, attempt 1 = fence-stripped retry
        text = raw if attempt == 0 else _strip_fences(raw)
        try:
            data = json.loads(text)
            return JudgeResponse.model_validate(data)
        except (json.JSONDecodeError, ValidationError):
            if attempt == 1:
                raise JudgeError(
                    "invalid JSON after salvage (prompt_id=judge_system)"
                ) from None
    raise JudgeError("invalid JSON after salvage (unreachable)")

result = invoke_judge(rendered_prompt, llm=llm)
print("judge judged:", result.winner, "|", result.score_a, "/", result.score_b)

judge judged: A | 9 / 2


## 8. Assert schema (exit criterion)

`winner` present + ∈ {"A","B"}, `score_a`/`score_b` present, `reasoning` non-empty.

In [12]:
# TODO: assert isinstance(result, JudgeResponse)
# TODO: print(result.model_dump_json(indent=2))  # stdout check for T-02-13

assert isinstance(result, JudgeResponse)
assert result.winner in {"A", "B"}           # D25 domain
assert 0 <= result.score_a <= 10               # D27 Field bounds
assert 0 <= result.score_b <= 10
assert len(result.reasoning) >= 1              # D27 min_length
assert result.winner == "A", (             # conflict: A surfaces drift, B misses it
    f"expected A to win (surfaces S1/S2 drift), got {result.winner}: "
    f"A={result.score_a} B={result.score_b}"
)
print(result.model_dump_json(indent=2))  # stdout check for T-02-13

{
  "reasoning": "Candidate A provides a detailed and accurate response that aligns closely with the ideal answer. It correctly identifies the inconsistency in the embedding models used in step 1 and step 3, mentioning the specific models and dimensions. Candidate A also correctly surfaces the model drift as required by the ideal answer. In contrast, Candidate B's response is incorrect and incomplete. It states that the shared pipeline uses the same embedding model everywhere, which contradicts the information provided in the ideal answer. Candidate B fails to mention the different embeddings used in steps 1 and 3, and incorrectly states that they are the same. Both responses are clear but only Candidate A is factually correct and complete.",
  "score_a": 9,
  "score_b": 2,
  "winner": "A"
}


## 9. Summary / promotion note

Modules are **promoted**: `src/llmops/config/judge.py` (§1–3, 7) and
`src/llmops/prompts/{__init__,registry}.py` (§4). NB-002 stays as the prototype; the
promoted modules are the source of truth (T-02-13a). §10 below exercises the **promoted**
lifecycle module on a scratch copy.


In [13]:
# TODO: print("NB-002 anchor check: judge wiring OK")

print("NB-002 anchor check: judge wiring OK")
print("promoted: src/llmops/config/judge.py + src/llmops/prompts/{__init__,registry}.py")
print("NB-002 anchor check complete; §10 lifecycle demo exercises the promoted module")

NB-002 anchor check: judge wiring OK
promoted: src/llmops/config/judge.py + src/llmops/prompts/{__init__,registry}.py
NB-002 anchor check complete; §10 lifecycle demo exercises the promoted module


## 10. Lifecycle demo — approve / rollback on a scratch copy (exit criterion 3)

D31: one approved version ships; the approve/rollback **mechanism** is proven by tests
T-02-7/8 AND this demo. Operates on `deepcopy(registry)` so the shipped
`registry.json` is never mutated. Uses the promoted module (source of truth, T-02-13a).


In [14]:
from copy import deepcopy

from llmops.prompts.registry import (
    approve as registry_approve,
)
from llmops.prompts.registry import (
    load_registry as registry_load,
)
from llmops.prompts.registry import (
    rollback as registry_rollback,
)
from llmops.prompts.registry import (
    select_approved as registry_select,
)

scratch = deepcopy(registry)      # never touches the shipped registry.json

registry_approve(scratch, "judge_system_generic_1.1.0")
assert scratch["prompts"]["judge_system_generic_1.1.0"]["status"] == "approved"
assert scratch["prompts"]["judge_system_generic_1.0.0"]["status"] == "retired"

registry_rollback(scratch, "judge_system_generic_1.1.0")
assert scratch["prompts"]["judge_system_generic_1.0.0"]["status"] == "approved"
assert scratch["prompts"]["judge_system_generic_1.1.0"]["status"] == "retired"

picked = registry_select(scratch, "judge_system", "generic")
assert picked["key"] == "judge_system_generic_1.0.0"

live = registry_load()   # the promoted loader reads the shipped file
assert live["prompts"]["judge_system_generic_1.0.0"]["status"] == "approved"
assert live["prompts"]["judge_system_generic_1.1.0"]["status"] == "draft"

print("lifecycle demo OK: approve->rollback->restore on scratch copy; shipped registry untouched")


lifecycle demo OK: approve->rollback->restore on scratch copy; shipped registry untouched
